# Notebook 5 — What-If Scenario Analysis & Cost Optimisation

**Series:** DES Playground · Discrete Event Simulation in SimPy  
**Prerequisites:** Notebooks 1–4

**What you'll learn:**
- Structure a **scenario comparison** framework with multiple alternatives
- Run all scenarios with the same seeds for fair statistical comparison
- Build a **cost model** that weighs server cost against customer-wait cost
- Use confidence intervals to determine when differences are **statistically significant**
- Make a data-driven capacity decision

---

## 1 · The Problem

A help-desk receives **12 calls per hour** (λ = 0.2/min).  Each call takes
on average **4 minutes** to handle (μ = 0.25/min).  Current setup: 1 agent.

Current utilisation ρ = λ/μ = **0.8** → mean wait Wq ≈ 16 min.  
The manager wants to evaluate four alternatives:

| Scenario | Description |
|----------|-------------|
| A (Baseline) | 1 agent, exponential service, mean 4 min |
| B | 2 agents (same service rate) |
| C | 1 agent, reduce mean service to 3 min (process improvement) |
| D | 1 agent, switch from exponential to constant 4-min service |

**Cost assumptions:**
- Server cost: €40 / hour per agent
- Customer wait cost: €10 / hour waiting (opportunity cost of blocked callers)

In [ ]:
!pip install simpy --quiet

In [ ]:
import simpy
import random
import math
import statistics
from dataclasses import dataclass
from typing import List, Dict, Callable, Optional
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

T_TABLE = {1:12.706,2:4.303,3:3.182,4:2.776,5:2.571,6:2.447,7:2.365,
           8:2.306,9:2.262,10:2.228,15:2.131,20:2.086,25:2.060,30:2.042}

def t_crit(df):
    if df in T_TABLE: return T_TABLE[df]
    if df > 30: return 1.960
    keys = sorted(T_TABLE)
    for i in range(len(keys)-1):
        lo, hi = keys[i], keys[i+1]
        if lo < df < hi:
            return T_TABLE[lo] + (T_TABLE[hi]-T_TABLE[lo])*(df-lo)/(hi-lo)
    return 2.0

def ci(values: List[float]) -> Dict:
    n = len(values)
    if n < 2: return dict(mean=values[0] if values else 0, half=0, lo=0, hi=0)
    mu = statistics.mean(values)
    s  = statistics.stdev(values)
    h  = t_crit(n-1) * s / math.sqrt(n)
    return dict(mean=mu, std=s, half=h, lo=mu-h, hi=mu+h)

print('Setup complete.')

## 2 · Generic Single-Queue Simulation

In [ ]:
def run_queue(
    arrival_rate:  float,
    svc_sampler:   Callable,   # fn(rng) → service time
    servers:       int,
    horizon:       float = 30_000,
    warmup:        float = 2_000,
    seed:          int   = 42,
) -> Dict:
    rng = random.Random(seed)
    env = simpy.Environment()
    srv = simpy.Resource(env, capacity=servers)

    last_t=[0.0]; q_area=[0.0]; b_area=[0.0]
    wait_t: List[float]=[]; soj_t: List[float]=[]
    deps=[0]; arrivals=[0]

    def integrate(t):
        dt=t-last_t[0]; q_area[0]+=len(srv.queue)*dt; b_area[0]+=srv.count*dt; last_t[0]=t

    def entity(arr_t):
        track=arr_t>=warmup
        integrate(env.now)
        arrivals[0]+=1
        q_in=env.now
        with srv.request() as req:
            yield req
            integrate(env.now)
            wait=env.now-q_in
            svc=svc_sampler(rng)
            yield env.timeout(svc)
            integrate(env.now)
        deps[0]+=1
        if track:
            wait_t.append(wait)
            soj_t.append(wait+svc)

    def source():
        while True:
            yield env.timeout(rng.expovariate(arrival_rate))
            env.process(entity(env.now))

    env.process(source())
    env.run(until=horizon)
    integrate(horizon)

    obs=max(horizon-warmup,1.0)
    return {
        'lq':         q_area[0]/obs,
        'rho':        (b_area[0]/obs)/servers,
        'wq':         statistics.mean(wait_t) if wait_t else 0.0,
        'w':          statistics.mean(soj_t)  if soj_t  else 0.0,
        'throughput': deps[0]/obs,
        'servers':    servers,
    }

print('run_queue() defined.')

## 3 · Define All Scenarios

In [ ]:
LAMBDA = 0.2    # calls per minute  (12 calls/hour)

scenarios = {
    'A — Baseline (1 agent, Exp 4min)': {
        'sampler':  lambda rng: rng.expovariate(1/4),
        'servers':  1,
        'agent_cost_per_hr': 40,
        'color': 'steelblue',
    },
    'B — 2 agents (Exp 4min)': {
        'sampler':  lambda rng: rng.expovariate(1/4),
        'servers':  2,
        'agent_cost_per_hr': 80,
        'color': 'mediumseagreen',
    },
    'C — 1 agent, faster (Exp 3min)': {
        'sampler':  lambda rng: rng.expovariate(1/3),
        'servers':  1,
        'agent_cost_per_hr': 40,
        'color': 'orange',
    },
    'D — 1 agent, constant 4min': {
        'sampler':  lambda rng: 4.0,
        'servers':  1,
        'agent_cost_per_hr': 40,
        'color': 'orchid',
    },
}

BASE_SEED = 42
N_REPS    = 10

print('Running scenarios...')
scenario_data = {}

for name, cfg in scenarios.items():
    runs = [
        run_queue(LAMBDA, cfg['sampler'], cfg['servers'],
                  horizon=40_000, warmup=2_000, seed=BASE_SEED+k*1000)
        for k in range(N_REPS)
    ]
    scenario_data[name] = {
        'cfg':  cfg,
        'lq':   ci([r['lq']  for r in runs]),
        'rho':  ci([r['rho'] for r in runs]),
        'wq':   ci([r['wq']  for r in runs]),
        'w':    ci([r['w']   for r in runs]),
    }
    wq = scenario_data[name]['wq']
    rho = scenario_data[name]['rho']
    print(f'  {name[:40]:<40}  Wq={wq["mean"]:6.2f}±{wq["half"]:4.2f}  ρ={rho["mean"]:.3f}')

print('Done.')

## 4 · Side-by-Side Comparison

In [ ]:
names_s = list(scenario_data.keys())
short_names = [n.split('—')[0].strip() for n in names_s]
colors_s = [scenario_data[n]['cfg']['color'] for n in names_s]
x = np.arange(len(names_s))

wq_m  = [scenario_data[n]['wq']['mean'] for n in names_s]
wq_h  = [scenario_data[n]['wq']['half'] for n in names_s]
lq_m  = [scenario_data[n]['lq']['mean'] for n in names_s]
rho_m = [scenario_data[n]['rho']['mean'] for n in names_s]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].bar(x, wq_m, color=colors_s, alpha=0.85,
            yerr=wq_h, error_kw=dict(capsize=5))
axes[0].set_xticks(x); axes[0].set_xticklabels(short_names, rotation=20, ha='right')
axes[0].set_ylabel('Mean wait Wq (minutes)')
axes[0].set_title('Queue wait time')

axes[1].bar(x, lq_m, color=colors_s, alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(short_names, rotation=20, ha='right')
axes[1].set_ylabel('Mean queue length Lq')
axes[1].set_title('Queue length')

axes[2].bar(x, rho_m, color=colors_s, alpha=0.85)
axes[2].axhline(1.0, color='red', ls='--', lw=1.5, label='ρ=1 (unstable)')
axes[2].set_xticks(x); axes[2].set_xticklabels(short_names, rotation=20, ha='right')
axes[2].set_ylabel('Utilisation ρ')
axes[2].set_title('Server utilisation')
axes[2].legend()

plt.suptitle('Help-Desk Scenario Comparison', fontsize=13)
plt.tight_layout()
plt.show()

## 5 · Cost Model

Total hourly cost = **server cost** + **customer wait cost**:

```
cost_per_hour = (servers × €40) + (Lq × €10)
```

`Lq` is the mean number of customers waiting in queue at any moment.  
By Little's Law, `Lq = λ × Wq` — paying €10/hr per waiting customer is
equivalent to paying €10 × (waiting fraction) of the hourly arrival volume.

In [ ]:
COST_SERVER_PER_HR  = 40.0   # € per server per hour
COST_WAIT_PER_HR    = 10.0   # € per customer-hour in queue

print(f'{"Scenario":<42} {"Server €/hr":>11} {"Wait €/hr":>10} {"Total €/hr":>11} {"Best"}')
print('-' * 82)

costs = {}
for name in names_s:
    d = scenario_data[name]
    srv_cost  = d['cfg']['servers'] * COST_SERVER_PER_HR
    wait_cost = d['lq']['mean'] * COST_WAIT_PER_HR
    total     = srv_cost + wait_cost
    costs[name] = total
    print(f'{name[:42]:<42} {srv_cost:>11.2f} {wait_cost:>10.2f} {total:>11.2f}')

best = min(costs, key=costs.get)
print(f'\n→ Lowest total cost: "{best}" at €{costs[best]:.2f}/hr')

In [ ]:
srv_costs  = [scenario_data[n]['cfg']['servers'] * COST_SERVER_PER_HR for n in names_s]
wait_costs = [scenario_data[n]['lq']['mean'] * COST_WAIT_PER_HR for n in names_s]
totals     = [s + w for s, w in zip(srv_costs, wait_costs)]

fig, ax = plt.subplots(figsize=(10, 5))

bars_s = ax.bar(x, srv_costs,  label='Server cost',      color='steelblue', alpha=0.85)
bars_w = ax.bar(x, wait_costs, bottom=srv_costs, label='Customer wait cost', color='coral', alpha=0.85)

# Annotate total
for i, (xi, total) in enumerate(zip(x, totals)):
    ax.text(xi, total + 0.5, f'€{total:.0f}', ha='center', fontsize=10, fontweight='bold')

# Highlight best
best_idx = totals.index(min(totals))
ax.get_children()[best_idx].set_edgecolor('gold')
ax.get_children()[best_idx].set_linewidth(3)

ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=15, ha='right')
ax.set_ylabel('Cost per hour (€)')
ax.set_title('Total cost breakdown — server cost + customer wait cost')
ax.legend()
plt.tight_layout()
plt.show()

## 6 · Statistical Significance of Differences

Before acting on the comparison, we check: is the difference between scenarios
**statistically significant**?  We use a paired t-test on the replication-level Wq values.

In [ ]:
# Re-collect per-replication Wq for pairwise testing
per_rep: Dict[str, List[float]] = {}
for name, cfg in scenarios.items():
    per_rep[name] = [
        run_queue(LAMBDA, cfg['sampler'], cfg['servers'],
                  horizon=40_000, warmup=2_000, seed=BASE_SEED+k*1000)['wq']
        for k in range(N_REPS)
    ]

baseline_name = names_s[0]
baseline_wq   = per_rep[baseline_name]

print(f'Pairwise comparison vs Baseline ("{baseline_name[:30]}")')
print(f'{"Alternative":<42} {"Δ Wq":>8} {"95% CI":>14} {"Significant?"}')
print('-' * 76)

for name in names_s[1:]:
    alt_wq = per_rep[name]
    diffs  = [b - a for b, a in zip(baseline_wq, alt_wq)]  # positive = improvement
    d_ci   = ci(diffs)
    sig    = 'YES ✓' if d_ci['lo'] > 0 else ('Borderline' if d_ci['lo'] > -0.5 else 'NO ✗')
    print(f'{name[:42]:<42} {d_ci["mean"]:>8.2f} [{d_ci["lo"]:>5.2f}, {d_ci["hi"]:>5.2f}]  {sig}')

print()
print('Interpretation: if the 95% CI for Δ Wq is entirely > 0, the improvement is statistically significant.')

## 7 · Sensitivity Analysis — Cost Ratio

The optimal scenario depends on the **ratio** of wait cost to server cost.
Let's sweep this ratio and see when each scenario becomes cheapest.

In [ ]:
wait_cost_ratios = np.linspace(0, 60, 200)   # wait cost per customer-hour in queue

fig, ax = plt.subplots(figsize=(10, 5))

for name in names_s:
    d   = scenario_data[name]
    srv = d['cfg']['servers'] * COST_SERVER_PER_HR
    lq  = d['lq']['mean']
    total_costs = [srv + lq * wc for wc in wait_cost_ratios]
    ax.plot(wait_cost_ratios, total_costs,
            label=name.split('—')[0].strip(),
            color=d['cfg']['color'], lw=2)

ax.axvline(COST_WAIT_PER_HR, color='black', ls=':', lw=1.5, label=f'Current wait cost = €{COST_WAIT_PER_HR}/hr')
ax.set_xlabel('Wait cost (€ per customer-hour in queue)')
ax.set_ylabel('Total cost per hour (€)')
ax.set_title('Sensitivity — which scenario is cheapest at different wait costs?')
ax.legend(fontsize=9)
ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

## 8 · Summary & Decision Framework

| Step | What to do |
|------|------------|
| **1 — Measure** | Collect real arrival and service time data; fit distributions |
| **2 — Build model** | Implement in SimPy; verify against theory (M/M/c if applicable) |
| **3 — Run scenarios** | Use same seeds across scenarios for fair comparison |
| **4 — Confidence intervals** | Report mean ± 95% CI for all KPIs; use ≥10 replications |
| **5 — Cost model** | Translate KPIs to business cost (server hours + wait cost) |
| **6 — Significance test** | Paired t-test on replication diffs — only act on significant changes |
| **7 — Sensitivity** | Vary the cost assumptions — does the best option remain best? |

---

## Series Complete!

You have now worked through the full DES Playground simulation curriculum:

| Notebook | Topic |
|----------|-------|
| [01](01_mm1_single_server.ipynb) | M/M/1 — single server, warm-up, CI, Little's Law |
| [02](02_mmc_multi_server.ipynb) | M/M/c — multiple servers, Erlang-C, SLA analysis |
| [03](03_md1_vs_mm1_variability.ipynb) | Variability — P-K formula, distribution comparison |
| [04](04_production_line.ipynb) | Production line — bottleneck, cascade effect |
| [05](05_what_if_analysis.ipynb) | What-if — scenario comparison, cost model, significance |

Open the [DES Playground](https://des-playground.example.com) to build and
visualise these same models interactively — the Python code generated by the
app uses exactly the same simulation engine as these notebooks.